# Aula 1 - Criando um agente

## Vídeo 1.2 - Entendendo a estrutura do Markdown

https://quarto.org/docs/presentations/

In [ ]:
%pip install pydantic-ai-slim[tavily]
%pip install docling
%pip install fastembed
%pip install qdrant-client

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
import os
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

In [ ]:
from pydantic_ai import Agent
import nest_asyncio
nest_asyncio.apply()

In [ ]:
agent = Agent(
    'google-gla:gemini-2.0-flash',
    system_prompt='''Você é um criador de apresentações que cria apresentações neste formato:
    ---
title: "Habits"
author: "John Doe"
format: revealjs
---

## Getting up

- Turn off alarm
- Get out of bed

## Going to sleep

- Get in bed
- Count sheep

Ao receber um tema crie a apresentação.


    ''',
)

In [ ]:
result = agent.run_sync('Indústria de petshops')

In [ ]:
print(result.data)

## Vídeo 1.3 - Formatando resultados

https://ai.pydantic.dev/results/#result-validators-functions

In [ ]:
from pydantic import BaseModel
import yaml  # Biblioteca para manipular YAML

In [ ]:
class Formatacao(BaseModel):
    title: str
    author: str
    format: str
    theme: str
    incremental: bool


In [ ]:
agente_formatador = Agent(
    'google-gla:gemini-2.0-flash',
    system_prompt='''Você é um criador de apresentações que retorna a formatação para apresentação. A formatação tem o seguinte formato:
---
title: "Presentation"
author: "John Doe"
format:
  revealjs:
    theme: dark
    incremental: true
---

As opções de tema são:
beige
blood
dark
default
league
moon
night
serif
simple
sky
solarized

Você deve inferir quais as propriedades da formatação a partir do prompt.
''',result_type=Formatacao
)

In [ ]:
#result = agent.run_sync('Quero uma apresentação com o tema escuro, o autor é a Gatito Petshop, e o título é Ganhos em vendas de arranhadores.')
#print(result.data)

In [ ]:
result = agente_formatador.run_sync('Quero uma apresentação com o tema escuro, o autor é a Gatito Petshop, e o título é Ganhos em vendas de arranhadores.')
print(result.data)

In [ ]:
result.data.model_dump()

In [ ]:
def formatar_para_yaml(result_data):
    """
    Formata os dados de result.data no formato YAML esperado.
    """
    # Constrói a estrutura do dicionário com base nos dados fornecidos
    yaml_data = {
        "title": result_data.title,
        "author": result_data.author,
        "format": {
            result_data.format: {
                "theme": result_data.theme,
                "incremental": result_data.incremental,
            }
        },
    }

    # Converte para YAML
    import yaml
    return yaml.dump(
        yaml_data,
        sort_keys=False,  # Mantém a ordem dos campos
        default_flow_style=False,  # Gera o YAML no estilo de múltiplas linhas
    )


In [ ]:
# Formata para YAML
yaml_formatado = formatar_para_yaml(result.data)

print(yaml_formatado)

## Vídeo 1.4 - Avaliando resultados

https://ai.pydantic.dev/results/#result-validators-functions

In [ ]:
from pydantic_ai import ModelRetry

In [ ]:
@agente_formatador.result_validator
async def valida_resultado(result):
  if result.title == 'Presentation':
      raise ModelRetry(f'''
      Você precisa passar um título que tenha relação com gatos e a apresentação de resultados semestrais da Gatito Petshop.''')
  else:
    return result


In [ ]:
result = agente_formatador.run_sync('Quero uma apresentação com o tema escuro, o autor é a Gatito Petshop.')

In [ ]:
result.data

In [ ]:
yaml_formatado = formatar_para_yaml(result.data)

print(yaml_formatado)